In [1]:
%pwd


'c:\\Users\\Vinay Verma\\Desktop\\code\\Skin-Disease-Detection\\research'

In [2]:
import os
os.chdir("../")

In [3]:
%pwd

'c:\\Users\\Vinay Verma\\Desktop\\code\\Skin-Disease-Detection'

In [4]:
import dagshub
dagshub.init(repo_owner='vinayverma07', repo_name='Skin-Disease-Detection-MLOPs', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

c:\Users\Vinay Verma\Desktop\code\Skin-Disease-Detection\.venv\Lib\site-packages\rich\live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=9c9919ea-0a64-479f-8124-ccc585129a3f&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=1ea8aa0fb6bed959f9222698ad37578878b371af07be8241f79953d7aa357d27




Accessing as vinayverma07

Initialized MLflow to track repo "vinayverma07/Skin-Disease-Detection-MLOPs"

Repository vinayverma07/Skin-Disease-Detection-MLOPs initialized!

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    history_path: Path
    plots_dir: Path
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [6]:
from src.cnnClassifier.constants import *
from src.cnnClassifier.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = self.config.model_evaluation
        
        

        create_directories([eval_config.root_dir, eval_config.plots_dir])

        evaluation_config= EvaluationConfig(
            root_dir=Path(eval_config.root_dir),
            test_data_path=Path(eval_config.test_data_path),
            model_path=Path(eval_config.model_path),
            history_path=Path(eval_config.history_path),
            plots_dir=Path(eval_config.plots_dir),
            mlflow_uri="https://dagshub.com/vinayverma07/Skin-Disease-Detection-MLOPs.mlflow",
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        
        return evaluation_config

In [7]:
import os
import json
import tensorflow as tf
import numpy as np
import mlflow
import mlflow.keras
from pathlib import Path
from dotenv import load_dotenv
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from src.cnnClassifier import logger
# from cnnClassifier.entity.config_entity import EvaluationConfig
from src.cnnClassifier.utils.visualization import generate_training_curves, generate_confusion_matrix_plot

load_dotenv()

class ModelEvaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def evaluate(self):
        """Runs validation calculations, generates assets, and tracking records for 22 skin disease classes."""
        # 1. Load data and model
        logger.info("Loading validation data and trained model assets...")
        test_ds = tf.data.Dataset.load(str(self.config.test_data_path))
        model = tf.keras.models.load_model(str(self.config.model_path))

        # 2. Extract predictions across all classes
        logger.info("Generating predictions across test dataset tensors...")
        y_true = []
        y_pred_probs = []

        for images, labels in test_ds:
            y_true.extend(np.argmax(labels.numpy(), axis=1))
            preds = model.predict(images, verbose=0)
            y_pred_probs.extend(preds)

        y_true = np.array(y_true)
        y_pred_probs = np.array(y_pred_probs)
        y_pred = np.argmax(y_pred_probs, axis=1)

        # 3. Compute Multi-Class Medical Metrics (Adapted for 22 classes)
        accuracy = np.mean(y_true == y_pred)
        
        # Using 'weighted' average to balance metrics according to the true support of each of the 22 classes
        precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
        recall = recall_score(y_true, y_pred, average='weighted', zero_division=0) # Sensitivity
        f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
        
        # Multi-class AUC-ROC calculation using the One-vs-Rest (OVR) strategy
        try:
            auc_roc = roc_auc_score(y_true, y_pred_probs, multi_class='ovr', average='weighted')
        except Exception as e:
            logger.warning(f"Could not calculate multi-class AUC-ROC (likely due to missing classes in the current batch slice): {e}")
            auc_roc = 0.0

        metrics_dict = {
            "val_accuracy": accuracy,
            "val_precision": precision,
            "val_recall_sensitivity": recall,
            "val_f1_score": f1,
            "val_auc_roc": auc_roc
        }
        
        params_dict = {
            "image_size": str(self.config.params_image_size),  # Convert list to string for MLflow compatibility
            "batch_size": self.config.params_batch_size,
            "num_classes": 22
        }

        # 4. Invoke Isolated Plotting Module
        logger.info("Calling internal plotting utilities to isolate diagnostic assets...")
        generate_training_curves(str(self.config.history_path), str(self.config.plots_dir))
        
        # Dynamically generate 22 class labels (e.g., Class_0 to Class_21)
        # If you have an explicit list of names, replace this with your list of 22 strings
        class_names = [f"Class_{i}" for i in range(22)]
        generate_confusion_matrix_plot(y_true, y_pred, classes=class_names, save_dir=str(self.config.plots_dir))

        # 5. Connect to MLflow remote server via DagsHub
        if self.config.mlflow_uri != "":
            mlflow.set_tracking_uri(self.config.mlflow_uri)
            
        # Ensure experiment exists for skin disease tracking
        mlflow.set_experiment("Skin_Disease_Classification_22_Classes")

        with mlflow.start_run():
            logger.info("Logging runtime parameters and diagnostic metrics to MLflow...")
            
            # Log Parameters
            mlflow.log_params(params_dict)
            
            # Log Metrics
            mlflow.log_metrics(metrics_dict)
            
            # Log Plots as Artifacts
            mlflow.log_artifacts(str(self.config.plots_dir), artifact_path="evaluation_plots")
            
            # Log Model Architecture
            mlflow.keras.log_model(model, "model", registered_model_name="Skin_Disease_EfficientNet_Model")
            
        logger.info("Evaluation metrics successfully tracked and artifacts archived.")

c:\Users\Vinay Verma\Desktop\code\Skin-Disease-Detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
STAGE_NAME = "Evaluation Stage"
try:
    logger.info(f">>>>>> stage {STAGE_NAME} started <<<<<<")
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    model_evaluation = ModelEvaluation(config=eval_config)
    model_evaluation.evaluate()
    logger.info(f">>>>>> stage {STAGE_NAME} completed <<<<<<\n\nx==========x")
except Exception as e:
    logger.exception(e)
    raise e